# Myanmar Voice Clone — Gradio Colab

Notebook này là bản **voice clone only** có giao diện Gradio.

Không dùng prompt voice preset `Myanmar 1–4`. Bạn chỉ chọn/upload **reference voice** thật, nhập text Myanmar, chỉnh thông số, rồi generate.

Luồng chạy:

1. Mount Google Drive.
2. Cài thư viện.
3. Tạo lại package pipeline từ các file `.py`.
4. Mở Gradio UI.
5. Chọn voice trong thư mục Drive hoặc upload voice mới.
6. Generate và nghe output trực tiếp.


In [10]:
#@title 1) Kiểm tra GPU và mount Google Drive
import os, sys, json, shutil, subprocess, textwrap, importlib, gc
from pathlib import Path

try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch chưa sẵn sàng:", e)

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DRIVE_DIR = Path("/content/drive/MyDrive/OmniVoice_Myanmar_Gradio")
REF_DIR = PROJECT_DRIVE_DIR / "reference_voices"
OUTPUT_DIR = PROJECT_DRIVE_DIR / "outputs"
CACHE_DIR = PROJECT_DRIVE_DIR / "voice_clone_cache"
WORK_DIR = PROJECT_DRIVE_DIR / "work"

for d in [PROJECT_DRIVE_DIR, REF_DIR, OUTPUT_DIR, CACHE_DIR, WORK_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_DRIVE_DIR:", PROJECT_DRIVE_DIR)
print("REF_DIR:", REF_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CACHE_DIR:", CACHE_DIR)


Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_DRIVE_DIR: /content/drive/MyDrive/OmniVoice_Myanmar_Gradio
REF_DIR: /content/drive/MyDrive/OmniVoice_Myanmar_Gradio/reference_voices
OUTPUT_DIR: /content/drive/MyDrive/OmniVoice_Myanmar_Gradio/outputs
CACHE_DIR: /content/drive/MyDrive/OmniVoice_Myanmar_Gradio/voice_clone_cache


In [11]:
#@title 2) Cài thư viện cần thiết
# Nếu pip install omnivoice lỗi, bật INSTALL_FROM_GITHUB = True rồi chạy lại cell.
INSTALL_FROM_GITHUB = False  #@param {type:"boolean"}

base_packages = [
    "soundfile",
    "librosa",
    "pedalboard",
    "gradio",
    "accelerate",
    "safetensors",
    "transformers",
    "huggingface_hub",
]

cmd = [sys.executable, "-m", "pip", "install", "-q", "-U"] + base_packages
print("Installing base packages...")
subprocess.check_call(cmd)

if INSTALL_FROM_GITHUB:
    print("Installing OmniVoice from GitHub...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "git+https://github.com/k2-fsa/OmniVoice.git"])
else:
    print("Installing OmniVoice from PyPI...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "omnivoice"])

print("Done.")


Installing base packages...
Installing OmniVoice from PyPI...
Done.


In [12]:
#@title 3) Tạo package pipeline từ code local để test notebook
from pathlib import Path
import shutil

PKG_ROOT = Path("/content/myanmar_voice_pipeline_src")
PKG_DIR = PKG_ROOT / "myanmar_voice_pipeline"
PKG_DIR.mkdir(parents=True, exist_ok=True)
(PKG_DIR / "__init__.py").write_text("", encoding="utf-8")

source_candidates = [
    Path.cwd() / "omnivoice_runpod_serverless" / "app",
    Path.cwd().parent / "omnivoice_runpod_serverless" / "app",
    Path("/content/OmniVoice Production/omnivoice_runpod_serverless/app"),
    Path("/content/drive/MyDrive/OmniVoice Production/omnivoice_runpod_serverless/app"),
]

source_root = next((path for path in source_candidates if path.exists()), None)
if source_root is None:
    raise FileNotFoundError(
        "Không tìm thấy thư mục omnivoice_runpod_serverless/app. "
        "Hãy mount đúng project trước khi chạy cell này."
    )

module_files = [
    "__init__.py",
    "audio_processing.py",
    "engine.py",
    "presets.py",
    "text_normalization.py",
]

for filename in module_files:
    shutil.copy2(source_root / filename, PKG_DIR / filename)

print(f"Copied package files from: {source_root}")
print("Files:", sorted(p.name for p in PKG_DIR.iterdir()))



FileNotFoundError: Không tìm thấy thư mục omnivoice_runpod_serverless/app. Hãy mount đúng project trước khi chạy cell này.

In [ ]:
#@title 4) Import pipeline và cấu hình model
import os, sys, json, time, tempfile, shutil, gc
from pathlib import Path

os.environ["OUTPUT_DIR"] = str(OUTPUT_DIR)
os.environ["PROMPT_CACHE_DIR"] = str(CACHE_DIR)  # cache embedding reference voice clone
os.environ["MODEL_ID"] = "k2-fsa/OmniVoice"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from myanmar_voice_pipeline.engine import OmniVoiceService
from myanmar_voice_pipeline.text_normalization import preprocess_text_for_tts, segment_text_with_pauses
from myanmar_voice_pipeline.presets import get_effective_config, clamp_prosody
from myanmar_voice_pipeline.audio_processing import preprocess_reference_audio, analyze_reference_audio, load_audio_mono

LANGUAGE = "my"

print("Pipeline imported.")
print("Model:", os.environ["MODEL_ID"])
print("Reference voice folder:", REF_DIR)


Pipeline imported.
Model: k2-fsa/OmniVoice
Reference voice folder: /content/drive/MyDrive/OmniVoice_Myanmar_Gradio/reference_voices


## 5) Thêm voice clone vào thư mục Drive

Bạn có 2 cách:

- Cách 1: Upload trực tiếp trong Gradio UI.
- Cách 2: Copy file `.wav/.mp3/.flac/.m4a/.ogg` vào thư mục:

`/content/drive/MyDrive/OmniVoice_Myanmar_Gradio/reference_voices`

Tên file nên dễ nhớ, ví dụ:

- `girl_clean_01.wav`
- `male_ads_01.wav`
- `myanmar_voice_test.wav`

Reference voice nên dài **3–10 giây**, 1 người nói, ít noise, không nhạc nền.


In [ ]:
#@title 6) Chạy Gradio UI để tune tiếng Việt trước khi update RunPod
import gradio as gr
import numpy as np
import soundfile as sf
from IPython.display import display
from pathlib import Path
import os, json, shutil, tempfile, time, traceback, gc

AUDIO_EXTS = {".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aac", ".webm"}
TEST_LANGUAGE = "vi"

_service = None

def get_service():
    global _service
    if _service is None:
        _service = OmniVoiceService(
            model_id=os.getenv("MODEL_ID", "k2-fsa/OmniVoice"),
            output_dir=str(OUTPUT_DIR),
            prompt_cache_dir=str(CACHE_DIR),
        )
        _service.load_model()
    return _service

def list_voice_files():
    REF_DIR.mkdir(parents=True, exist_ok=True)
    files = []
    for p in sorted(REF_DIR.iterdir()):
        if p.is_file() and p.suffix.lower() in AUDIO_EXTS:
            files.append(p.name)
    return files

def refresh_voice_dropdown():
    choices = list_voice_files()
    value = choices[0] if choices else None
    return gr.update(choices=choices, value=value), f"Đã tìm thấy {len(choices)} voice trong Drive."

def safe_voice_filename(name: str, original_path: str):
    name = (name or "").strip()
    ext = Path(original_path).suffix.lower() or ".wav"
    if not name:
        name = Path(original_path).stem
    keep = []
    for ch in name:
        if ch.isalnum() or ch in ("-", "_", " "):
            keep.append(ch)
    clean = "".join(keep).strip().replace(" ", "_")
    if not clean:
        clean = f"voice_{int(time.time())}"
    if not clean.lower().endswith(ext):
        clean += ext
    return clean

def save_uploaded_voice(uploaded_audio, voice_name):
    if not uploaded_audio:
        choices = list_voice_files()
        return gr.update(choices=choices, value=(choices[0] if choices else None)), "Bạn chưa upload audio."
    src = Path(uploaded_audio)
    if not src.exists():
        choices = list_voice_files()
        return gr.update(choices=choices, value=(choices[0] if choices else None)), f"Không tìm thấy file upload: {src}"

    filename = safe_voice_filename(voice_name, str(src))
    dst = REF_DIR / filename
    shutil.copy2(src, dst)

    choices = list_voice_files()
    return gr.update(choices=choices, value=filename), f"Đã lưu voice vào Drive: {dst}"

def get_reference_path(selected_voice, uploaded_audio, prefer_uploaded):
    if prefer_uploaded and uploaded_audio:
        return str(uploaded_audio), "uploaded_audio"
    if selected_voice:
        p = REF_DIR / selected_voice
        if p.exists():
            return str(p), "drive_voice"
    if uploaded_audio:
        return str(uploaded_audio), "uploaded_audio"
    raise ValueError("Bạn cần chọn voice trong Drive hoặc upload reference audio.")

def preview_reference(selected_voice, uploaded_audio, prefer_uploaded):
    try:
        ref_path, source = get_reference_path(selected_voice, uploaded_audio, prefer_uploaded)
        audio, sr = load_audio_mono(ref_path, sr=None)
        meta = analyze_reference_audio(audio, sr)
        return ref_path, json.dumps({"source": source, "path": ref_path, "quality": meta}, ensure_ascii=False, indent=2)
    except Exception as e:
        return None, f"Lỗi preview reference: {e}"

def normalize_preview(text, join_silence_ms, max_segment_chars):
    try:
        normalized = preprocess_text_for_tts(text or "", TEST_LANGUAGE, is_reference=False)
        chunks = segment_text_with_pauses(
            normalized,
            join_silence_ms=int(join_silence_ms),
            max_chars=int(max_segment_chars) if max_segment_chars else None,
            min_chars=None,
            lang=TEST_LANGUAGE,
        )
        return normalized, json.dumps(chunks, ensure_ascii=False, indent=2)
    except Exception as e:
        return "", f"Lỗi normalize/segment: {e}"

def generate_voice_clone(
    text,
    selected_voice,
    uploaded_audio,
    prefer_uploaded,
    reference_text,
    emotion,
    ad_emphasis,
    num_step,
    guidance_scale,
    speed,
    pitch_shift,
    join_silence_ms,
    trailing_silence_ms,
    max_segment_chars,
    preprocess_reference,
    ref_apply_vad,
    ref_vad_top_db,
    ref_apply_denoise,
    ref_denoise_strength,
    ref_apply_rms_normalize,
    ref_target_rms_dbfs,
):
    try:
        text = (text or "").strip()
        if not text:
            raise ValueError("Bạn chưa nhập text tiếng Việt cần đọc.")

        ref_path, ref_source = get_reference_path(selected_voice, uploaded_audio, prefer_uploaded)
        ref_text = (reference_text or "").strip() or None

        service = get_service()
        output_name = f"vietnamese_clone_{int(time.time())}.wav"

        result = service.synthesize(
            text=text,
            language=TEST_LANGUAGE,
            mode="clone",
            reference_audio_path=ref_path,
            ref_text=ref_text,
            emotion=emotion,
            ad_emphasis=ad_emphasis,
            ad_safe=True,
            speed=float(speed),
            pitch_shift=float(pitch_shift),
            num_step=int(num_step),
            guidance_scale=float(guidance_scale),
            join_silence_ms=int(join_silence_ms),
            trailing_silence_ms=int(trailing_silence_ms),
            max_segment_chars=int(max_segment_chars) if max_segment_chars else None,
            output_filename=output_name,
            return_base64=False,
            save_output=True,
            debug=True,
            preprocess_reference=bool(preprocess_reference),
            ref_trim_silence=True,
            ref_trim_top_db=35,
            ref_apply_vad=bool(ref_apply_vad),
            ref_vad_top_db=int(ref_vad_top_db),
            ref_max_internal_silence_ms=120,
            ref_apply_denoise=bool(ref_apply_denoise),
            ref_denoise_strength=float(ref_denoise_strength),
            ref_apply_rms_normalize=bool(ref_apply_rms_normalize),
            ref_target_rms_dbfs=float(ref_target_rms_dbfs),
            ref_min_seconds=1.5,
            ref_max_seconds=10.0,
            work_dir=str(WORK_DIR / f"job_{int(time.time())}"),
        )

        out_path = result.get("output_path")
        normalized = result.get("text", result.get("normalized_text", ""))
        chunks = result.get("segments", result.get("chunks", []))

        meta = {
            "reference_source": ref_source,
            "reference_path": ref_path,
            "output_path": out_path,
            "config": result.get("config"),
            "effective_config": result.get("effective_config"),
            "reference": result.get("reference"),
            "compute_seconds": result.get("compute_seconds"),
            "segments": chunks,
            "reference_quality": result.get("reference_quality"),
        }

        return out_path, normalized, json.dumps(meta, ensure_ascii=False, indent=2)

    except Exception as e:
        err = traceback.format_exc()
        return None, "", f"ERROR: {e}\n\n{err}"

def unload_model():
    global _service
    _service = None
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    return "Đã unload model và dọn GPU cache."

initial_choices = list_voice_files()
initial_value = initial_choices[0] if initial_choices else None

with gr.Blocks(title="Vietnamese Voice Clone Tuning", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # Vietnamese Voice Clone Tuning

    Dùng notebook này để test riêng **tiếng Việt** trước khi update RunPod.
    Bạn có thể nghe A/B bằng cách chỉnh `join_silence_ms`, `trailing_silence_ms`, `max_segment_chars`, `speed`, `pitch_shift`.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## 1) Chọn voice reference")
            selected_voice = gr.Dropdown(
                choices=initial_choices,
                value=initial_value,
                label="Voice trong Google Drive",
                info="Các file nằm trong thư mục reference_voices.",
            )
            refresh_btn = gr.Button("Reload danh sách voice")
            refresh_status = gr.Textbox(label="Trạng thái", interactive=False)

            uploaded_audio = gr.Audio(
                label="Hoặc upload reference voice mới",
                type="filepath",
                sources=["upload", "microphone"],
            )
            prefer_uploaded = gr.Checkbox(
                value=True,
                label="Ưu tiên dùng audio vừa upload khi generate",
            )
            voice_name = gr.Textbox(
                label="Tên lưu voice vào Drive",
                placeholder="VD: vietnamese_female_clean_01",
            )
            save_voice_btn = gr.Button("Lưu uploaded voice vào Drive")

            preview_btn = gr.Button("Kiểm tra reference voice")
            ref_audio_preview = gr.Audio(label="Reference đang dùng", type="filepath")
            ref_meta = gr.Textbox(label="Thông tin reference", lines=10)

        with gr.Column(scale=2):
            gr.Markdown("## 2) Text và thông số clone")
            text_input = gr.Textbox(
                label="Text tiếng Việt cần đọc",
                lines=6,
                value="Khám phá giải pháp đột phá giúp nâng tầm cuộc sống của bạn ngay hôm nay. Sản phẩm chất lượng vượt trội, thiết kế tinh tế cùng ưu đãi hấp dẫn đang chờ đón bạn sở hữu.",
            )
            reference_text = gr.Textbox(
                label="Ref text / transcript của reference voice",
                lines=4,
                value="Chúng tôi tin rằng, mỗi bữa ăn ngon chính là sợi dây kết nối hạnh phúc gia đình. Hãy để chúng tôi cùng bạn chăm sóc những người thân yêu nhất mỗi ngày.",
                placeholder="Paste transcript đúng của voice mẫu vào đây. Có thể để trống nếu không có.",
                interactive=True,
            )

            with gr.Accordion("Thông số chính", open=True):
                with gr.Row():
                    num_step = gr.Slider(24, 48, value=32, step=1, label="Num step")
                    guidance_scale = gr.Slider(1.0, 5.0, value=3.9, step=0.05, label="Guidance")
                with gr.Row():
                    speed = gr.Slider(0.85, 1.3, value=1.0, step=0.005, label="Speed")
                    pitch_shift = gr.Slider(0.95, 1.08, value=1.0, step=0.005, label="Pitch")
                with gr.Row():
                    join_silence_ms = gr.Slider(0, 180, value=110, step=5, label="Join silence ms")
                    trailing_silence_ms = gr.Slider(80, 400, value=220, step=5, label="Trailing silence ms")
                    max_segment_chars = gr.Slider(80, 220, value=160, step=5, label="Max segment chars")

            with gr.Accordion("Style nhẹ / cảm xúc", open=False):
                emotion = gr.Dropdown(
                    ["Mặc định", "Vui vẻ (Happy)", "Buồn bã (Sad)", "Hào hứng (Excited)", "Giận dữ (Angry)", "Nhẹ nhàng (Gentle)"],
                    value="Mặc định",
                    label="Emotion",
                )
                ad_emphasis = gr.Dropdown(
                    ["Không bổ trợ", "Cường điệu rất nhẹ", "Cường điệu nhẹ", "Cường điệu vừa", "Cường điệu mạnh"],
                    value="Không bổ trợ",
                    label="Ad emphasis",
                )

            with gr.Accordion("Preprocess reference audio", open=False):
                preprocess_reference = gr.Checkbox(value=True, label="Bật preprocess reference")
                ref_apply_vad = gr.Checkbox(value=True, label="VAD / cắt khoảng lặng")
                ref_vad_top_db = gr.Slider(24, 45, value=32, step=1, label="VAD top_db")
                ref_apply_denoise = gr.Checkbox(value=False, label="Denoise nhẹ")
                ref_denoise_strength = gr.Slider(0.05, 0.35, value=0.18, step=0.01, label="Denoise strength")
                ref_apply_rms_normalize = gr.Checkbox(value=True, label="RMS normalize")
                ref_target_rms_dbfs = gr.Slider(-30, -16, value=-22, step=0.5, label="Target RMS dBFS")

            with gr.Row():
                normalize_btn = gr.Button("Preview normalize/segment")
                generate_btn = gr.Button("Generate clone voice", variant="primary")
                unload_btn = gr.Button("Unload model")

            normalized_text = gr.Textbox(label="Normalized text", lines=6)
            segment_preview = gr.Textbox(label="Segment preview / metadata", lines=14)
            output_audio = gr.Audio(label="Kết quả audio", type="filepath")

    refresh_btn.click(refresh_voice_dropdown, inputs=[], outputs=[selected_voice, refresh_status])
    save_voice_btn.click(save_uploaded_voice, inputs=[uploaded_audio, voice_name], outputs=[selected_voice, refresh_status])
    preview_btn.click(preview_reference, inputs=[selected_voice, uploaded_audio, prefer_uploaded], outputs=[ref_audio_preview, ref_meta])

    normalize_btn.click(
        normalize_preview,
        inputs=[text_input, join_silence_ms, max_segment_chars],
        outputs=[normalized_text, segment_preview],
    )

    generate_btn.click(
        generate_voice_clone,
        inputs=[
            text_input,
            selected_voice,
            uploaded_audio,
            prefer_uploaded,
            reference_text,
            emotion,
            ad_emphasis,
            num_step,
            guidance_scale,
            speed,
            pitch_shift,
            join_silence_ms,
            trailing_silence_ms,
            max_segment_chars,
            preprocess_reference,
            ref_apply_vad,
            ref_vad_top_db,
            ref_apply_denoise,
            ref_denoise_strength,
            ref_apply_rms_normalize,
            ref_target_rms_dbfs,
        ],
        outputs=[output_audio, normalized_text, segment_preview],
    )

    unload_btn.click(unload_model, inputs=[], outputs=[refresh_status])

demo.queue(max_size=10)
demo.launch(share=True, debug=True)




/tmp/ipykernel_2264/1398165416.py:208: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Myanmar Voice Clone Only", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6fe30053336783f900.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

## Gợi ý chỉnh khi test

- Nếu có tiếng ngắt kiểu “è/ừ”: thử giảm `Guidance` từ `4.0` xuống `3.8–3.9`, giữ `Num step = 36`.
- Nếu phát âm chưa rõ chữ: thử tăng `Guidance` lên `4.1–4.25` hoặc `Num step = 40`.
- Nếu giọng bị robotic/cứng: giảm `Guidance`, giữ `Speed = 1.0; có thể kéo tối đa 2.0 nếu cần nhanh hơn`, tắt `Ad emphasis`.
- Nếu câu dài bị vỡ nhịp: giảm `Max segment chars` xuống `70–80`.
- Nếu nối câu quá gấp: tăng `Join silence ms` lên `110–130`.
- Reference voice nên sạch, 3–10 giây, một người nói, không nhạc nền.
